# Quantization (binning): turning counts into buckets

_Feature Engineering (Zheng & Casari)_

**Chop a heavy-tailed count into a handful of bins so the big values stop dominating.**

Some raw numbers span a huge range. A business on Yelp might have 3 reviews or
       3,000. A song might have been played twice or two million times. When a feature like this
       feeds a model, the giant values shout over the small ones: a linear model treats the
       difference between 1 and 2 reviews the same as between 2,001 and 2,002, even though the first
       difference matters far more.

       Quantization (also called binning) fixes this. You stop using the raw count and
       instead say which bucket it falls into: "0&ndash;9 reviews", "10&ndash;99 reviews", and so
       on. The model now sees a small ordered category instead of an unbounded number. This is the very
       first transform in Chapter 2 of Zheng & Casari's Feature Engineering for Machine Learning,
       and their running example is the Yelp business review_count &mdash; a classic
       heavy-tailed count.

---

This notebook is a practice scaffold for the **Quantization (binning): turning counts into buckets** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas + numpy

In [ ]:
import numpy as np
import pandas as pd
import json

# --- Load the Yelp business data (Yelp Dataset Challenge, round 6) ---
# Get it from the book's repo: github.com/alicezheng/feature-engineering-book
biz_file = open('yelp_academic_dataset_business.json')
biz_df = pd.DataFrame([json.loads(x) for x in biz_file.readlines()])
biz_file.close()

# 'review_count' is a heavy-tailed count: most businesses have a few reviews,
# a handful have thousands.

# === Fixed-width binning ===
# Linear: map a count to a bin by INTEGER DIVISION (bin width = 10).
small_counts = np.array([0, 5, 13, 28, 37, 99, 100, 7000])
np.floor_divide(small_counts, 10)
# -> array([  0,   0,   1,   2,   3,   9,  10, 700])

# Exponential: group by powers of 10 (take the log10, then floor).
large_counts = np.array([296, 8286, 64011, 80, 3, 725, 867, 2215,
                         7689, 11495, 91897, 44, 28, 7971, 926, 122, 22222])
np.floor(np.log10(large_counts))
# -> array([2., 3., 4., 1., 0., 2., 2., 3., 3., 4., 4., 1., 1., 3., 2., 2., 4.])

# === Quantile binning (adaptive: equal-count bins) ===
# Cut review_count into 4 equal-count bins; labels=False -> integer bin index.
pd.qcut(large_counts, 4, labels=False)

# Read the deciles straight off the data to see where the edges fall:
deciles = biz_df['review_count'].quantile([.1, .2, .3, .4, .5, .6, .7, .8, .9])
print(deciles)
# For the Yelp review_count the deciles came out:
# 0.1 -> 3   0.2 -> 4   0.3 -> 5   0.4 -> 6   0.5 -> 8
# 0.6 -> 12  0.7 -> 17  0.8 -> 28  0.9 -> 58
# Tight at the low end, then jumping to 58 -- exactly mirroring the heavy tail.

## Visualize the data & results

_Take a skewed real count (the 'mean area' of cells in load_breast_cancer) and split it into 5 fixed-width bins vs 5 quantile bins. Which scheme keeps the bins evenly populated?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

d = load_breast_cancer()                         # 569 real cell-nucleus measurements
fi = list(d.feature_names).index('mean area')
area = d.data[:, fi]                              # right-skewed, count-like feature

rng = np.random.RandomState(0)
area = np.sort(area[rng.choice(len(area), 60, replace=False)])   # subsample to 60

# Fixed-width: 5 equal-width bins across the range (numpy only, no pandas).
fw_edges = np.linspace(area.min(), area.max(), 6)
fw_occ = np.histogram(area, bins=fw_edges)[0]
print('fixed-width edges  :', np.round(fw_edges).astype(int))
print('fixed-width counts :', fw_occ)            # -> [17 25  7  5  6]  (lopsided)

# Quantile: edges at the 0,20,40,60,80,100 percentiles -> equal-count bins.
q_edges = np.percentile(area, [0, 20, 40, 60, 80, 100])
q_occ = np.histogram(area, bins=q_edges)[0]
print('quantile edges     :', np.round(q_edges).astype(int))
print('quantile counts    :', q_occ)             # -> [12 12 12 12 12]  (even)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You bin the Yelp review_count into 20 equal-width bins. Almost every business lands in bin 0 and bins 5 through 19 are empty. What went wrong and what are the two fixes from the chapter?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize the feature is heavy-tailed: a few businesses have thousands of reviews, but the median has a handful. — _Equal-width bins of width $w$ put nearly all the mass in the first bin and leave the tail bins empty._
- Switch to exponential bins: $\lfloor\log_{10}x\rfloor$ via np.floor(np.log10(...)). — _Power-of-10 bands are ten times wider each step, so the long tail gets a few wide bins instead of hundreds of empty ones._
- Or switch to quantile bins via pd.qcut, cutting at the deciles. — _Equal-count bins guarantee every bucket is populated, regardless of how skewed the raw counts are._

**Answer:** The empty-bin problem: equal-width bins over a heavy tail dump everyone into bin 0 and leave the rest empty. The chapter's two fixes are exponential (log) binning &mdash; np.floor(np.log10(x)) &mdash; and quantile binning &mdash; pd.qcut(x, n, labels=False), whose deciles for the Yelp data were 3, 4, 5, 6, 8, 12, 17, 28, 58.

</details>

**Problem 2.** A teammate computes quantile bin edges on the full dataset, then splits into train and test. Why is this a problem, and how should the edges be produced instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that quantile edges are learned from data &mdash; they are parameters of the transform. — _Unlike a fixed width of 10, the decile cut points come from the values themselves._
- Computing them on the full dataset uses test-set values to set the edges. — _That is data leakage: information from the test set has bled into the feature definition, inflating measured performance._
- Fit the quantiles on the training set only, store the edges, and apply those same edges to validation/test/production. — _This keeps the transform honest and reproducible, exactly like fitting any other preprocessor on train only._

**Answer:** Quantile edges are learned parameters, so computing them on all the data leaks test information into the feature. Fit the deciles on the training set only, save those edges, and reuse them everywhere. Fixed-width edges (a constant width like 10) don't have this issue because they don't depend on the sample.

</details>

**Problem 3.** For two different features &mdash; a 1-to-5 star rating, and a play-count that ranges from 2 to 4 million &mdash; which binning scheme fits each, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look at the range and shape of each feature. — _The star rating is small and bounded; the play-count is heavy-tailed across orders of magnitude._
- For the bounded, roughly-uniform rating, use fixed-width linear bins (or leave it as is). — _Equal-width bins are simple and the buckets stay populated because the range is small and even._
- For the play-count, use exponential (log) bins or quantile bins. — _Equal-width bins would be almost all empty; log bands or equal-count quantiles match the heavy tail._

**Answer:** The bounded, uniform-ish star rating suits fixed-width linear bins. The heavy-tailed play-count needs an adaptive scheme: exponential (log) binning for fixed, data-independent power-of-10 bands, or quantile binning for guaranteed equal-count buckets. Equal-width bins on the play-count would be mostly empty.

</details>